# VM Resource Planner Walkthrough

Notebook này thực thi pipeline `vm_resource_planner.py` theo từng bước để quan sát rõ Phase 1→2→3 khi chuyển kết quả forecast thành kế hoạch VM allocation.

Các bước chính:
1. Load cấu hình & các mô hình forecast mới nhất
2. Sinh forecast trên tập test
3. Chuyển forecast thành nhu cầu CPU/RAM cần offload khỏi máy hiện tại
4. Giải bài toán phân bổ VM bằng Linear Programming cho hai kịch bản
5. Lập lịch VM theo từng time-bucket và lưu kết quả



In [ ]:
import json
from pathlib import Path

import pandas as pd

from vm_resource_planner import (
    load_vm_catalog,
    load_latest_models,
    generate_forecasts,
    convert_forecasts_to_requirements,
    summarize_peak_plans,
    build_schedule,
    HOST_SPEC,
    VM_TYPES_FILE,
    RESULTS_DIR,
    SUPPORTED_MODELS,
)

MODEL_NAME = "hybrid_prophet_lstm"  # thay đổi sang 'random_forest', 'svr', 'arimax' nếu muốn
RESULTS_DIR.mkdir(exist_ok=True)
print("✓ Libraries & planner helpers loaded")
print("Available models:", list(SUPPORTED_MODELS.keys()))
print("Current model:", MODEL_NAME)


✓ Libraries & planner helpers loaded


In [ ]:
vm_catalog = load_vm_catalog(VM_TYPES_FILE)
model_paths = load_latest_models(MODEL_NAME)

print("VM catalog:")
for spec in vm_catalog:
    print(f"  • {spec['name']}: {spec['vcpus']} vCPUs, {spec['memory_gb']} GB, ${spec['cost_per_hour']}/h")

print("\nLatest model checkpoints:")
for target, path in model_paths.items():
    print(f"  {target}: {path}")

HOST_SPEC


In [ ]:
forecast_df = generate_forecasts(model_paths, MODEL_NAME)
forecast_df.head()


In [ ]:
requirements_df = convert_forecasts_to_requirements(forecast_df, HOST_SPEC)
requirements_df[['timestamp','cpu_total_usage','cpu_required_cores','cpu_overflow_cores','memory_usage_pct','memory_required_gb','memory_overflow_gb']].head()


In [ ]:
peak_summary = summarize_peak_plans(requirements_df, vm_catalog)
peak_summary


In [ ]:
schedule_df = build_schedule(requirements_df, vm_catalog)
schedule_df.head()


In [ ]:
report_path = RESULTS_DIR / "vm_resource_planning_notebook.json"
schedule_path = RESULTS_DIR / "vm_schedule_notebook.csv"

schedule_records = schedule_df.copy()
schedule_records['timestamp'] = schedule_records['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')

report_payload = {
    'model': 'vm_resource_planner_notebook',
    'host_spec': HOST_SPEC,
    'peak_summary': peak_summary,
    'schedule': schedule_records.to_dict(orient='records'),
}

with open(report_path, 'w') as f:
    json.dump(report_payload, f, indent=2)

schedule_df.to_csv(schedule_path, index=False)

report_path, schedule_path
